# Monitor Service

> Service for job execution monitoring with resource telemetry.

In [ ]:
#| default_exp services.monitor

In [ ]:
#| export
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.queue import JobQueue, Job, JobStatus
from cjm_plugin_system.core.manager import PluginManager

from cjm_fasthtml_job_monitor.models import ResourceSnapshot

## JobMonitorService

Wraps `JobQueue` interactions and resource telemetry into a single service.
Consumers create one instance and pass it to the route factory.

In [ ]:
#| export
class JobMonitorService:
    """Service for job execution monitoring with resource telemetry."""

    def __init__(
        self,
        queue: JobQueue,                          # Job queue instance
        manager: PluginManager,                   # For worker stats, logs, sysmon
        sysmon_plugin_name: Optional[str] = None, # System monitor plugin name (e.g., 'cjm-system-monitor-nvidia')
    ):
        self._queue = queue
        self._manager = manager
        self._sysmon_name = sysmon_plugin_name

### Job Operations

In [ ]:
#| export
async def _submit_job(
    self,
    plugin_name: str,       # Target plugin
    *args,
    priority: int = 0,
    **kwargs
) -> str:  # job_id
    """Submit a job to the queue."""
    return await self._queue.submit(plugin_name, *args, priority=priority, **kwargs)

def _get_job(self, job_id: str) -> Optional[Job]:  # Job or None
    """Get job by ID."""
    return self._queue.get_job(job_id)

async def _cancel_job(self, job_id: str) -> bool:  # True if cancelled
    """Cancel a job."""
    return await self._queue.cancel(job_id)

JobMonitorService.submit_job = _submit_job
JobMonitorService.get_job = _get_job
JobMonitorService.cancel_job = _cancel_job

### Log Tailing

In [ ]:
#| export
def _get_logs(
    self,
    plugin_name: str,              # Plugin whose logs to read
    lines: int = 50,               # Max lines to return
    current_session_only: bool = True,  # Filter to current session
) -> str:  # Log text
    """Get plugin logs, optionally filtered to current session."""
    # Read extra lines so session filtering has enough to work with
    read_lines = lines * 3 if current_session_only else lines
    raw = self._manager.get_plugin_logs(plugin_name, lines=read_lines)
    if current_session_only:
        return _filter_current_session(raw, lines)
    return raw

JobMonitorService.get_logs = _get_logs

In [ ]:
#| export
def _filter_current_session(
    raw: str,        # Full log text
    max_lines: int,  # Max lines to return
) -> str:  # Filtered log text
    """Extract logs from the most recent session (after last '--- Starting' marker)."""
    all_lines = raw.split('\n')
    last_marker = -1
    for i, line in enumerate(all_lines):
        if line.strip().startswith('--- Starting'):
            last_marker = i
    if last_marker >= 0:
        session_lines = all_lines[last_marker + 1:]
    else:
        session_lines = all_lines
    # Strip trailing empty lines
    while session_lines and not session_lines[-1].strip():
        session_lines.pop()
    return '\n'.join(session_lines[-max_lines:])

In [ ]:
# Test _filter_current_session
multi_session_log = """--- Starting plugin-a at Mon Mar 24 10:00:00 2026 ---
Old session line 1
Old session line 2
--- Starting plugin-a at Mon Mar 24 12:00:00 2026 ---
Current session line 1
Current session line 2
Current session line 3
"""

result = _filter_current_session(multi_session_log, max_lines=50)
assert "Old session" not in result
assert "Current session line 1" in result
assert "Current session line 2" in result
assert "Current session line 3" in result
assert "--- Starting" not in result
print(f"Filtered to current session ({len(result.split(chr(10)))} lines)")
print(result)

Filtered to current session (3 lines)
Current session line 1
Current session line 2
Current session line 3


In [ ]:
# Test _filter_current_session with max_lines limit
long_session = "--- Starting plugin at now ---\n" + "\n".join(f"Line {i}" for i in range(100))
result = _filter_current_session(long_session, max_lines=5)
lines = result.strip().split('\n')
assert len(lines) == 5
assert "Line 99" in result
assert "Line 95" in result
assert "Line 94" not in result
print(f"Max lines limit works: got {len(lines)} lines, last is '{lines[-1]}'")

Max lines limit works: got 5 lines, last is 'Line 99'


In [ ]:
# Test _filter_current_session with no marker (returns tail of full log)
no_marker_log = "Line A\nLine B\nLine C\n"
result = _filter_current_session(no_marker_log, max_lines=50)
assert "Line A" in result
assert "Line B" in result
assert "Line C" in result
print("No marker case: returns all lines")

No marker case: returns all lines


### Resource Monitoring

In [ ]:
#| export
def _get_resource_snapshot(
    self,
    plugin_name: str,  # Plugin whose worker to query
) -> Optional[ResourceSnapshot]:  # Snapshot or None
    """Get current resource usage for a plugin's worker."""
    proxy = self._manager.get_plugin(plugin_name)
    if not proxy or not hasattr(proxy, 'get_stats'):
        return None

    try:
        stats = proxy.get_stats()
    except Exception:
        return None

    snapshot = ResourceSnapshot(
        worker_pid=stats.get('pid', 0),
        cpu_percent=stats.get('cpu_percent', 0.0),
        memory_rss_mb=stats.get('memory_rss_mb', 0.0),
    )

    # Enrich with GPU stats from system monitor
    _enrich_gpu_stats(self, snapshot)
    return snapshot

JobMonitorService.get_resource_snapshot = _get_resource_snapshot

In [ ]:
#| export
def _enrich_gpu_stats(
    self,
    snapshot: ResourceSnapshot,  # Snapshot to enrich in place
) -> None:
    """Add per-process GPU stats from system monitor plugin."""
    if not self._sysmon_name:
        return

    sysmon = self._manager.get_plugin(self._sysmon_name)
    if not sysmon:
        return

    try:
        sys_stats = sysmon.execute("get_system_status")
    except Exception:
        return

    details = sys_stats.get('details', {})
    processes = details.get('processes', [])

    # Find GPU process matching worker PID
    for proc in processes:
        if proc.get('pid') == snapshot.worker_pid:
            snapshot.gpu_memory_mb = proc.get('gpu_memory_mb', 0.0)
            snapshot.gpu_index = proc.get('gpu_index')
            break

    # Add global GPU info for context
    gpu_details = details.get('details', {})
    if snapshot.gpu_index is not None:
        gpu_key = f'gpu_{snapshot.gpu_index}'
        gpu = gpu_details.get(gpu_key, {})
        snapshot.gpu_name = gpu.get('name')
        snapshot.gpu_total_mb = gpu.get('memory_total')
        snapshot.gpu_load_percent = gpu.get('utilization')

JobMonitorService._enrich_gpu_stats = _enrich_gpu_stats

### Resource Snapshot Tests

These tests use mock objects to verify the enrichment logic without requiring actual plugins.

In [ ]:
# Test _enrich_gpu_stats with mock sysmon data
class MockSysmon:
    def execute(self, command):
        return {
            'cpu_percent': 15.0,
            'memory_used_mb': 8000.0,
            'memory_total_mb': 64000.0,
            'gpu_type': 'NVIDIA',
            'details': {
                'available': True,
                'type': 'NVIDIA',
                'details': {
                    'gpu_0': {
                        'name': 'NVIDIA GeForce RTX 4090',
                        'memory_total': 24564,
                        'memory_used': 3000,
                        'memory_free': 21564,
                        'utilization': 42,
                    }
                },
                'processes': [
                    {'pid': 99999, 'gpu_index': 0, 'gpu_memory_mb': 1200, 'command': 'python worker.py'},
                    {'pid': 88888, 'gpu_index': 0, 'gpu_memory_mb': 500, 'command': 'firefox'},
                ],
            }
        }

class MockManager:
    def __init__(self, sysmon=None):
        self._sysmon = sysmon
    def get_plugin(self, name):
        if name == 'sysmon' and self._sysmon:
            return self._sysmon
        return None
    def get_plugin_logs(self, name, lines=50):
        return "Log line 1\nLog line 2\n"

# Test enrichment with matching PID
service = JobMonitorService(
    queue=None,  # Not used for this test
    manager=MockManager(sysmon=MockSysmon()),
    sysmon_plugin_name='sysmon',
)
snap = ResourceSnapshot(worker_pid=99999, cpu_percent=30.0, memory_rss_mb=1500.0)
service._enrich_gpu_stats(snap)
assert snap.gpu_memory_mb == 1200
assert snap.gpu_index == 0
assert snap.gpu_name == 'NVIDIA GeForce RTX 4090'
assert snap.gpu_total_mb == 24564
assert snap.gpu_load_percent == 42
print(f"GPU enriched: {snap.gpu_memory_mb}MB on {snap.gpu_name} (load {snap.gpu_load_percent}%)")

GPU enriched: 1200MB on NVIDIA GeForce RTX 4090 (load 42%)


In [ ]:
# Test enrichment with non-matching PID (GPU fields stay None)
snap_no_gpu = ResourceSnapshot(worker_pid=11111, cpu_percent=10.0, memory_rss_mb=500.0)
service._enrich_gpu_stats(snap_no_gpu)
assert snap_no_gpu.gpu_memory_mb is None
assert snap_no_gpu.gpu_index is None
assert snap_no_gpu.gpu_name is None
print("Non-matching PID: GPU fields remain None")

Non-matching PID: GPU fields remain None


In [ ]:
# Test enrichment with no sysmon configured
service_no_sysmon = JobMonitorService(
    queue=None,
    manager=MockManager(),
    sysmon_plugin_name=None,
)
snap_no_sysmon = ResourceSnapshot(worker_pid=99999, cpu_percent=30.0, memory_rss_mb=1500.0)
service_no_sysmon._enrich_gpu_stats(snap_no_sysmon)
assert snap_no_sysmon.gpu_memory_mb is None
print("No sysmon configured: GPU fields remain None")

No sysmon configured: GPU fields remain None


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()